In [1]:
from __future__ import annotations

import os
from pathlib import Path
from typing import Generator, Optional, cast

os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

import torch
import torch.nn as nn
import torch.nn.functional as F
from bitsandbytes import nn as bnb_nn
from bitsandbytes import optim
from datasets import Dataset, load_dataset
from torch.amp import GradScaler, autocast
from torchao.quantization import Int4WeightOnlyConfig, quantize_
from torchinfo import summary
from tqdm.notebook import tqdm
from transformers import AutoTokenizer

from zoo.utils import get_device

Skipping import of cpp extensions due to incompatible torch version 2.8.0+cu128 for torchao version 0.16.0             Please see https://github.com/pytorch/ao/issues/2919 for more info


In [2]:
torch.set_float32_matmul_precision("medium")
device = get_device()

In [3]:
ds = load_dataset("karanravindra/simple-wikipedia")
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3-8B")

In [ ]:
def precompute_rope_for_liger(
    head_dim: int,
    max_seq_length: int,
    device: torch.device,
    theta: float = 10000.0,
    dtype: torch.dtype = torch.float32,
):
    """
    Precompute RoPE cos/sin for Liger.

    Liger expects cos/sin shaped:
      (1, seq_len, head_dim) or (bsz, seq_len, head_dim)

    So we compute half-dim frequencies, then expand to full head_dim.
    """
    assert head_dim % 2 == 0, "head_dim must be even for RoPE"

    inv_freq = 1.0 / (
        theta
        ** (torch.arange(0, head_dim, 2, device=device, dtype=torch.float32) / head_dim)
    )  # (head_dim // 2,)

    t = torch.arange(max_seq_length, device=device, dtype=torch.float32)  # (seq_len,)
    freqs = torch.outer(t, inv_freq)  # (seq_len, head_dim // 2)

    cos = freqs.cos()
    sin = freqs.sin()

    # Expand half-dim -> full head_dim for HF/Liger-style rotary application
    cos = torch.repeat_interleave(cos, 2, dim=-1)  # (seq_len, head_dim)
    sin = torch.repeat_interleave(sin, 2, dim=-1)  # (seq_len, head_dim)

    # Liger accepts (1, seq_len, head_dim)
    cos = cos.unsqueeze(0).to(dtype=dtype)
    sin = sin.unsqueeze(0).to(dtype=dtype)
    return cos, sin


def rotate_half(x: torch.Tensor) -> torch.Tensor:
    """
    Rotate half the dimensions for RoPE.

    For even head_dim, splits in half and swaps with a sign change:
    x = [x1, x2] -> [-x2, x1]

    This is the standard way to apply RoPE rotations.
    """
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat([-x2, x1], dim=-1)


def apply_rope_liger(
    q: torch.Tensor,  # (batch, num_heads, seq_len, head_dim)
    k: torch.Tensor,  # (batch, num_kv_heads, seq_len, head_dim)
    cos: torch.Tensor,  # (1, max_seq_len, head_dim)
    sin: torch.Tensor,  # (1, max_seq_len, head_dim)
):
    seq_length = q.shape[2]
    cos = cos[:, :seq_length, :].to(device=q.device, dtype=q.dtype)
    sin = sin[:, :seq_length, :].to(device=q.device, dtype=q.dtype)
    q = (q * cos) + (rotate_half(q) * sin)
    k = (k * cos) + (rotate_half(k) * sin)
    return q, k


class GroupQueryAttention(nn.Module):
    def __init__(
        self,
        dim: int,
        num_heads: int,
        num_kv_heads: Optional[int],
        max_seq_length: int,
    ):
        super().__init__()
        assert dim % num_heads == 0, "dim must be divisible by num_heads"

        if num_kv_heads is None:
            num_kv_heads = num_heads

        assert num_heads % num_kv_heads == 0, (
            "num_heads must be divisible by num_kv_heads"
        )

        self.num_heads = num_heads
        self.num_kv_heads = num_kv_heads
        self.head_dim = dim // num_heads
        self.num_groups = num_heads // num_kv_heads
        self.max_seq_length = max_seq_length

        self.q_proj = nn.Linear(dim, num_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(dim, num_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(dim, num_kv_heads * self.head_dim, bias=False)
        self.out_proj = nn.Linear(num_heads * self.head_dim, dim, bias=False)

        self.register_buffer("rope_cos", None, persistent=False)
        self.register_buffer("rope_sin", None, persistent=False)
        self._rope_cached = False

    def _init_rope(self, device: torch.device, dtype: torch.dtype):
        if not self._rope_cached:
            cos, sin = precompute_rope_for_liger(
                head_dim=self.head_dim,
                max_seq_length=self.max_seq_length,
                device=device,
                dtype=dtype,
            )
            self.rope_cos = cos
            self.rope_sin = sin
            self._rope_cached = True

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch_size, seq_length, _ = x.shape

        self._init_rope(x.device, x.dtype)

        q = self.q_proj(x).view(batch_size, seq_length, self.num_heads, self.head_dim)
        k = self.k_proj(x).view(
            batch_size, seq_length, self.num_kv_heads, self.head_dim
        )
        v = self.v_proj(x).view(
            batch_size, seq_length, self.num_kv_heads, self.head_dim
        )

        q = q.permute(0, 2, 1, 3)  # (B, Hq, T, D)
        k = k.permute(0, 2, 1, 3)  # (B, Hkv, T, D)
        v = v.permute(0, 2, 1, 3)  # (B, Hkv, T, D)

        q, k = apply_rope_liger(
            q,
            k,
            cast(torch.Tensor, self.rope_cos),
            cast(torch.Tensor, self.rope_sin),
        )

        attn_output = F.scaled_dot_product_attention(
            q, k, v, is_causal=True, enable_gqa=True
        )

        attn_output = attn_output.permute(0, 2, 1, 3).contiguous()
        attn_output = attn_output.view(batch_size, seq_length, -1)
        return self.out_proj(attn_output)


class MLP(nn.Module):
    """
    Same architecture as your SwiGLU MLP, but uses Liger's fused SiLU * multiply kernel.
    """

    def __init__(self, dim: int, multiplier: int = 4):
        super().__init__()
        hidden = dim * multiplier
        self.fc_gate = nn.Linear(dim, hidden, bias=False)
        self.fc_up = nn.Linear(dim, hidden, bias=False)
        self.fc_down = nn.Linear(hidden, dim, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc_down(F.silu(self.fc_gate(x)) * self.fc_up(x))


class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        norm_x = x.norm(dim=-1, keepdim=True) / (x.shape[-1] ** 0.5)
        return x / (norm_x + self.eps) * self.weight


class TransformerBlock(nn.Module):
    def __init__(
        self,
        dim: int,
        num_heads: int,
        num_kv_heads: Optional[int] = None,
        mlp_multiplier: int = 4,
        max_seq_length: int = 512,
    ):
        super().__init__()
        self.attention = GroupQueryAttention(
            dim, num_heads, num_kv_heads, max_seq_length
        )
        self.mlp = MLP(dim, mlp_multiplier)

        # Replace custom RMSNorm with LigerRMSNorm
        self.norm1 = RMSNorm(dim)
        self.norm2 = RMSNorm(dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attention(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


class Model(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        dim: int,
        num_blocks: int,
        num_heads: int,
        mlp_multiplier: int = 4,
        num_kv_heads: Optional[int] = None,
        max_seq_length: int = 512,
    ):
        super().__init__()
        self.num_blocks = num_blocks
        self.token_embedding = nn.Embedding(vocab_size, dim)

        self.blocks = nn.ModuleList(
            [
                TransformerBlock(
                    dim,
                    num_heads,
                    num_kv_heads,
                    mlp_multiplier,
                    max_seq_length,
                )
                for _ in range(num_blocks)
            ]
        )

        self.ln_f = RMSNorm(dim, eps=1e-6)
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
        elif isinstance(module, RMSNorm):
            if module.weight is not None:
                nn.init.ones_(module.weight)

        if isinstance(module, GroupQueryAttention):
            nn.init.normal_(
                module.out_proj.weight, std=0.02 / (2 * self.num_blocks) ** 0.5
            )
        if isinstance(module, MLP):
            nn.init.normal_(
                module.fc_down.weight, std=0.02 / (2 * self.num_blocks) ** 0.5
            )

    def forward_hidden(self, input_ids: torch.Tensor) -> torch.Tensor:
        x = self.token_embedding(input_ids)

        for block in self.blocks:
            x = block(x)

        x = self.ln_f(x)
        return x  # (B, T, D)

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        hidden = self.forward_hidden(input_ids)
        logits = F.linear(hidden, self.token_embedding.weight)  # (B, T, vocab_size)
        return logits


model = Model(
    vocab_size=len(tokenizer),
    dim=128,
    num_blocks=4,
    num_heads=16,
    num_kv_heads=4,  # GQA: 4 query heads per KV head
    max_seq_length=512,
).to(device)

summary(model, input_size=(1, 512), dtypes=[torch.long], device=device)

In [ ]:
optimizer = optim.AdamW8bit(model.parameters(), lr=8e-4)
criterion = nn.CrossEntropyLoss(ignore_index=-100)
scaler = GradScaler()
# qat_quantizer = Int8DynActInt4WeightQATQuantizer()
# model = qat_quantizer.prepare(model)

In [ ]:
def _dataset_cache_tag(dataset: Dataset) -> str:
    """
    Best-effort stable-ish identifier for a Hugging Face Dataset.
    Falls back to object id if no fingerprint is available.
    """
    fp = getattr(dataset, "_fingerprint", None)
    if fp:
        return fp
    # fallback: try features + length (not perfect, but better than nothing)
    try:
        return f"len{len(dataset)}_feat{hash(str(dataset.features)) & 0xFFFFFFFF:x}"
    except Exception:
        return f"obj{hex(id(dataset))}"


def _tokenizer_cache_tag(tokenizer: AutoTokenizer) -> str:
    """
    Prefer the tokenizer name/path when available; otherwise use class name.
    """
    name = getattr(tokenizer, "name_or_path", None)
    if name:
        # sanitize for filenames
        return "".join(c if c.isalnum() or c in "._-" else "_" for c in name)
    return tokenizer.__class__.__name__


def get_token_stream(
    dataset: Dataset,
    tokenizer: AutoTokenizer,
    seq_len: int,
    cache_dir: str | os.PathLike = ".cache/token_streams",
    cache_name: str | None = None,
    force_recompute: bool = False,
) -> tuple[list[int], list[list[int]]]:
    """Concatenates all documents into one long stream of tokens with EOS separators.

    Caches (stream, chunks) to a file after computation. If cache exists, loads it.
    """
    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)

    eos = tokenizer.eos_token_id
    if eos is None:
        raise ValueError("Tokenizer has no eos_token_id; cannot insert EOS separators.")

    if cache_name is None:
        ds_tag = _dataset_cache_tag(dataset)
        tok_tag = _tokenizer_cache_tag(tokenizer)
        cache_name = f"stream_chunks__ds-{ds_tag}__tok-{tok_tag}__seq-{seq_len}.pt"

    cache_path = cache_dir / cache_name

    if cache_path.exists() and not force_recompute:
        obj = torch.load(cache_path, map_location="cpu", weights_only=False)
        # Backward/forward compatibility checks
        if not isinstance(obj, dict) or "stream" not in obj or "chunks" not in obj:
            raise ValueError(f"Cache file {cache_path} is not in the expected format.")
        return obj["stream"], obj["chunks"]

    # Compute
    all_tokens = dataset["tokens"]  # fetch entire column at once
    stream = [tok for doc in all_tokens for tok in (*doc, eos)]

    chunks = [
        stream[i : i + seq_len + 1] for i in range(0, len(stream) - seq_len, seq_len)
    ]

    # Save (atomic-ish write)
    tmp_path = cache_path.with_suffix(cache_path.suffix + ".tmp")
    torch.save(
        {
            "stream": stream,
            "chunks": chunks,
            "meta": {
                "seq_len": seq_len,
                "eos_token_id": eos,
                "dataset_tag": _dataset_cache_tag(dataset),
                "tokenizer_tag": _tokenizer_cache_tag(tokenizer),
            },
        },
        tmp_path,
    )
    tmp_path.replace(cache_path)

    return stream, chunks


def get_batch(stream: list[int], chunks: list[list[int]], batch_size: int) -> Generator:
    """Generates batches from a packed token stream."""
    print(f"Total tokens: {len(stream):,}, total chunks: {len(chunks):,}")

    for i in range(0, len(chunks) - batch_size + 1, batch_size):
        batch_tensor = torch.tensor(chunks[i : i + batch_size], dtype=torch.long)
        yield {
            "input_ids": batch_tensor[:, :-1],
            "output_ids": batch_tensor[:, 1:],
        }


# get a few batches to check
train_stream, train_chunks = get_token_stream(
    ds["train"], tokenizer, seq_len=512, cache_dir=".cache/token_streams"
)
val_stream, val_chunks = get_token_stream(
    ds["val"], tokenizer, seq_len=512, cache_dir=".cache/token_streams"
)

train_batches = get_batch(train_stream, train_chunks, batch_size=8)
for _ in range(3):
    batch = next(train_batches)
    print(batch["input_ids"].shape, batch["output_ids"].shape)

In [ ]:
n_epochs = 1
for epoch in range(n_epochs):
    model.train()
    pbar = tqdm(
        get_batch(train_stream, train_chunks, batch_size=16),
        total=len(train_chunks) // 16,
        desc=f"Epoch {epoch + 1}/{n_epochs}",
    )

    for batch in pbar:
        optimizer.zero_grad(set_to_none=True)

        with autocast(device_type=device.type, dtype=torch.bfloat16):
            outputs = model(batch["input_ids"].to(device))  # (B, T, vocab_size)
            loss = criterion(
                outputs.view(-1, outputs.size(-1)),
                batch["output_ids"].to(device).view(-1),
            )

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        pbar.set_postfix({"loss": float(loss.item())})

    model.eval()
    val_loss = 0.0
    n_val_steps = 0

    with torch.no_grad():
        for batch in get_batch(val_stream, val_chunks, batch_size=16):
            inputs = batch["input_ids"].to(device)
            targets = batch["output_ids"].to(device)

            with autocast(device_type=device.type, dtype=torch.bfloat16):
                outputs = model(inputs)  # (B, T, vocab_size)
                loss = criterion(
                    outputs.view(-1, outputs.size(-1)), targets.view(-1)
                )  # flatten for cross-entropy

            val_loss += float(loss.item())
            n_val_steps += 1

    avg_val_loss = val_loss / max(n_val_steps, 1)
    print(f"Epoch {epoch + 1}/{n_epochs}, Validation Loss: {avg_val_loss:.4f}")

In [ ]:
# save the model
torch.save(model.state_dict(), "model.pth")

In [ ]:
# print model gradients
print("name, grad_norm, weight_norm, ratio, zeros")
for name, param in model.named_parameters():
    if param.grad is not None:
        grad_norm = param.grad.norm().item()
        weight_norm = param.norm().item()
        ratio = grad_norm / (weight_norm + 1e-6)
        zeros = (param.grad == 0).sum().item()
        print(f"{name}, {grad_norm:.4e}, {weight_norm:.4e}, {ratio:.4e}, {zeros}")

In [ ]:
print(f"Max memory: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

In [ ]:
@torch.inference_mode()
def generate(
    model, tokenizer, prompt, max_length=256, temperature=1, top_k=50, top_p=0.9
):
    model.eval()

    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
    generated_ids = input_ids

    for _ in range(max_length - input_ids.size(1)):
        outputs = model(generated_ids)
        next_token_logits = outputs[:, -1, :] / temperature

        # Top-k filtering
        if top_k > 0:
            top_k_values, _ = torch.topk(next_token_logits, top_k)
            next_token_logits[next_token_logits < top_k_values[:, -1, None]] = -float(
                "inf"
            )

        # Top-p (nucleus) filtering
        if top_p < 1.0:
            sorted_logits, sorted_indices = torch.sort(
                next_token_logits, descending=True, dim=-1
            )
            cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
            sorted_indices_to_remove = cumulative_probs > top_p
            sorted_indices_to_remove[:, -1] = False  # keep at least one token

            # Map mask back to original index space
            indices_to_remove = torch.zeros_like(
                next_token_logits, dtype=torch.bool, device=next_token_logits.device
            )
            indices_to_remove.scatter_(-1, sorted_indices, sorted_indices_to_remove)
            next_token_logits[indices_to_remove] = -float("inf")

        next_token_dist = torch.distributions.Categorical(logits=next_token_logits)
        next_token_id = next_token_dist.sample().unsqueeze(-1)
        generated_ids = torch.cat([generated_ids, next_token_id], dim=-1)

    return tokenizer.decode(generated_ids[0], skip_special_tokens=True)


print(generate(model, tokenizer, "{{infobox", max_length=50))

In [ ]:
test_stream, test_chunks = get_token_stream(ds["test"], tokenizer, seq_len=512)
# evaluate on test set
with torch.no_grad():
    model.eval().to(device)
    test_loss = 0.0
    for batch in tqdm(
        get_batch(test_stream, test_chunks, batch_size=16),
        total=len(test_chunks) // 16,
        desc="Testing",
    ):
        inputs = batch["input_ids"].to(device)
        targets = batch["output_ids"].to(device)

        outputs = model(inputs)
        loss = criterion(outputs.view(-1, outputs.size(-1)), targets.view(-1))
        test_loss += loss.item()
    avg_test_loss = test_loss / (len(test_chunks) // 16)
print(f"Test Loss: {avg_test_loss:.4f}")

## HellaSwag Evaluation

In [ ]:
@torch.inference_mode()
def batched_score_completions(
    model: nn.Module,
    tokenizer,
    ctx_texts: list[str],
    ending_texts: list[str],
    device: torch.device,
    max_len: int,
) -> torch.Tensor:
    """
    Scores (ctx_texts[i], ending_texts[i]) pairs in a single batch.
    Returns: losses shape (batch,) = mean NLL over completion tokens only.
    """
    assert len(ctx_texts) == len(ending_texts)
    bs = len(ctx_texts)

    # Tokenize separately so we know completion lengths
    ctx_enc = tokenizer(ctx_texts, add_special_tokens=False)
    end_enc = tokenizer(ending_texts, add_special_tokens=False)

    ctx_ids_list = ctx_enc["input_ids"]
    end_ids_list = end_enc["input_ids"]
    end_lens = torch.tensor([len(x) for x in end_ids_list], device=device)

    # Build concatenated sequences
    seqs = []
    for c_ids, e_ids in zip(ctx_ids_list, end_ids_list):
        seq = c_ids + e_ids
        # truncate from the left to fit max_len (same behavior as your code)
        if len(seq) > max_len:
            seq = seq[-max_len:]
        seqs.append(torch.tensor(seq, dtype=torch.long))

    # Pad to batch
    pad_id = tokenizer.pad_token_id
    if pad_id is None:
        # Many GPT-like tokenizers have no pad token by default
        pad_id = tokenizer.eos_token_id

    input_ids = torch.nn.utils.rnn.pad_sequence(
        seqs, batch_first=True, padding_value=pad_id
    )
    attention_mask = (input_ids != pad_id).long()

    input_ids = input_ids.to(device, non_blocking=True)
    attention_mask = attention_mask.to(device, non_blocking=True)

    # Forward pass once
    out = (
        model(input_ids, attention_mask=attention_mask)
        if "attention_mask" in model.forward.__code__.co_varnames
        else model(input_ids)
    )
    logits = out.logits if hasattr(out, "logits") else out  # (bs, T, V)

    # Next-token prediction shift
    # logits[:, t] predicts input_ids[:, t+1]
    log_probs = F.log_softmax(logits[:, :-1, :], dim=-1)  # (bs, T-1, V)
    targets = input_ids[:, 1:]  # (bs, T-1)

    # token-level NLL
    token_nll = -log_probs.gather(dim=-1, index=targets.unsqueeze(-1)).squeeze(
        -1
    )  # (bs, T-1)

    # Build mask for completion tokens only (aligned to targets positions)
    # We want the last end_len tokens of the sequence to be scored.
    # Let L = unpadded seq length after truncation.
    # targets positions correspond to tokens 1..L-1 in input_ids.
    seq_lens = attention_mask.sum(dim=1)  # (bs,)
    Tm1 = targets.size(1)
    pos = torch.arange(Tm1, device=device).unsqueeze(0).expand(bs, -1)  # 0..T-2

    # last valid target index is (L-2). completion targets are the last end_len tokens:
    # target indices in [L-1-end_len, L-2] (inclusive)
    start = (seq_lens - 1 - end_lens).clamp_min(0)  # (bs,)
    end = (seq_lens - 2).clamp_min(-1)  # (bs,)

    comp_mask = (pos >= start.unsqueeze(1)) & (pos <= end.unsqueeze(1))

    # Also exclude padding positions (safety)
    comp_mask = comp_mask & (pos < (seq_lens - 1).unsqueeze(1))

    # mean NLL over completion tokens
    denom = comp_mask.sum(dim=1).clamp_min(1)
    losses = (token_nll * comp_mask).sum(dim=1) / denom  # (bs,)
    return losses


def evaluate_hellaswag(
    model: nn.Module,
    tokenizer,
    num_examples: int | None = None,
    examples_per_batch: int = 16,  # <-- bump until you saturate GPU / hit memory
) -> float:
    hs_ds = load_dataset("Rowan/hellaswag", split="validation")
    if num_examples is not None:
        hs_ds = hs_ds.select(range(num_examples))

    torch.cuda.reset_peak_memory_stats()
    model.eval()

    # Determine model max len (your logic preserved)
    max_len = (
        model.position_embedding.num_embeddings
        if hasattr(model, "position_embedding")
        else 512
    )

    correct = 0
    n = len(hs_ds)

    # Micro-batch examples
    for i in tqdm(range(0, n, examples_per_batch), desc="HellaSwag (batched)"):
        batch = hs_ds[i : i + examples_per_batch]

        ctxs = batch["ctx"]
        endings_all = batch["endings"]  # list of 4-ending lists
        labels = list(map(int, batch["label"]))

        # Flatten to (B*4) pairs
        flat_ctxs, flat_ends = [], []
        for ctx, ends in zip(ctxs, endings_all):
            for e in ends:
                flat_ctxs.append(ctx)
                flat_ends.append(e)

        losses = batched_score_completions(
            model=model,
            tokenizer=tokenizer,
            ctx_texts=flat_ctxs,
            ending_texts=flat_ends,
            device=device,
            max_len=max_len,
        )  # (B*4,)

        # Unflatten to (B, 4)
        B = len(ctxs)
        losses = losses.view(B, 4)

        preds = losses.argmin(dim=1).tolist()  # lowest loss = best
        for p, y in zip(preds, labels):
            correct += int(p == y)

    acc = correct / n
    print(f"HellaSwag accuracy: {correct}/{n} = {acc:.4f}")

    # ---- SciPy "binomial" fix ----
    # One-sided p-value vs chance (0.25 for 4 options)
    bt = scipy.stats.binomtest(correct, n, p=0.25, alternative="greater")
    print(f"Binomtest p-value (vs 0.25, one-sided): {bt.pvalue:.3e}")

    # 95% CI for accuracy (exact Clopper-Pearson by default in recent SciPy)
    ci = scipy.stats.binomtest(correct, n).proportion_ci(confidence_level=0.95)
    print(f"Accuracy 95% CI: [{ci.low:.4f}, {ci.high:.4f}]")

    print(
        f"Max memory during evaluation: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB"
    )
    return acc


hellaswag_acc = evaluate_hellaswag(
    model, tokenizer, num_examples=1000, examples_per_batch=16
)